# enumerate
`enumerate` solves a common control-flow problem cleanly: you need both the item and its position. In real systems, positions matter for line numbers, batch offsets, rankings, and precise error reporting.

## 1. Problem First
Why does this matter in real systems?
- CSV and log parsing often needs line numbers for debugging.
- Batch processors may need positional offsets for retries or metrics.
- Index-based loops become noisy when you only need item plus position.

In [1]:
lines = ["INFO boot", "WARN disk", "ERROR db"]
for line_number, line in enumerate(lines, start=1):
    print(line_number, line)

1 INFO boot
2 WARN disk
3 ERROR db


## 2. Minimal Concept
Core syntax:
- `enumerate(iterable)`
- `enumerate(iterable, start=1)`
- It yields pairs: `(index, item)`.

## 3. Mental Model
How Python actually behaves
`enumerate` wraps an iterable and adds a counter as values are produced. It does not require precomputing indexes or converting the data structure. That means it preserves direct iteration while still giving you positional context.

In [2]:
items = ["api", "worker", "scheduler"]
for index, item in enumerate(items):
    print(index, item)

0 api
1 worker
2 scheduler


## 4. Internal Mechanics
`enumerate` returns an iterator object that internally tracks a counter and pulls values from the wrapped iterable. It is lightweight and fits naturally into `for` loops.

In [3]:
import dis

def show_positions(values):
    for index, value in enumerate(values, start=1):
        print(index, value)

dis.dis(show_positions)
print(list(enumerate(["a", "b"], start=5)))

  3           0 RESUME                   0

  4           2 LOAD_GLOBAL              1 (NULL + enumerate)
             12 LOAD_FAST                0 (values)
             14 LOAD_CONST               1 (1)
             16 KW_NAMES                 2 (('start',))
             18 CALL                     2
             26 GET_ITER
        >>   28 FOR_ITER                17 (to 66)
             32 UNPACK_SEQUENCE          2
             36 STORE_FAST               1 (index)
             38 STORE_FAST               2 (value)

  5          40 LOAD_GLOBAL              3 (NULL + print)
             50 LOAD_FAST                1 (index)
             52 LOAD_FAST                2 (value)
             54 CALL                     2
             62 POP_TOP
             64 JUMP_BACKWARD           19 (to 28)

  4     >>   66 END_FOR
             68 RETURN_CONST             0 (None)
[(5, 'a'), (6, 'b')]


## 5. Edge Cases
Where it breaks:
- Forgetting `start=1` can create off-by-one line numbers in human-facing output.
- Using `enumerate` when the index has no meaning adds noise.
- If you mutate the underlying list during iteration, the position labels can become misleading.

In [4]:
rows = ["header", "row1", "row2"]
for line_number, row in enumerate(rows):
    print(f"line {line_number}: {row}")

for line_number, row in enumerate(rows, start=1):
    print(f"human line {line_number}: {row}")

line 0: header
line 1: row1
line 2: row2
human line 1: header
human line 2: row1
human line 3: row2


## 6. Performance Thinking
- `enumerate` is O(1) extra memory.
- It is usually cleaner than `range(len(sequence))` when you need both index and value.
- Performance differences are small; readability and correctness are the bigger wins.

## 7. Real Use Case
A config parser can report the exact line number of malformed entries while still iterating directly over the input lines.

In [5]:
config_lines = ["host=api", "bad_line", "port=8080"]
for line_number, line in enumerate(config_lines, start=1):
    if "=" not in line:
        print(f"invalid config at line {line_number}: {line}")

invalid config at line 2: bad_line


## 8. Anti-Pattern
What beginners do wrong:
- Use `range(len(items))` for everything.
- Carry an extra manual counter variable when `enumerate` would be clearer.
- Expose zero-based indexes in user-facing messages without thinking about readability.

In [6]:
services = ["api", "worker"]
for i in range(len(services)):
    print(i, services[i])

for i, service in enumerate(services):
    print(i, service)

0 api
1 worker
0 api
1 worker


## 9. Interview Signals
Questions FAANG asks:
- Why prefer `enumerate` over `range(len(sequence))`?
- What does the `start` parameter solve?
- When is the index actually meaningful?
- How would you use `enumerate` in debugging or parsing workflows?

## 10. Exercise (Non-trivial)
Build a parser for a config or CSV-like file that validates each line, reports line-numbered errors, and preserves enough context to retry or fix only the bad rows. Decide whether line numbers should be zero-based for machines or one-based for humans.

In [7]:
def validate_lines(lines):
    errors = []
    for line_number, line in enumerate(lines, start=1):
        if "=" not in line:
            errors.append((line_number, line))
    return errors

# Task:
# 1. Add structured error reasons.
# 2. Decide whether start=1 is correct.
# 3. Explain why enumerate is better than manual counters here.